# Day 41: What AI, ML, DL and GenAI Are

**Today's question:** what is the difference between a rule *we write* and a rule a computer *learns from examples*?

No new Python today: only `csv`, functions and loops you already know. Before running a cell marked 🤔, predict what it will show.

## 1. The nested circles

```text
┌────────────────────────────────────────────────────────────────────┐
│ ARTIFICIAL INTELLIGENCE (AI): computers doing tasks we would       │
│ call "intelligent": hand-written rules, search, planning, and...   │
│  ┌──────────────────────────────────────────────────────────────┐  │
│  │ MACHINE LEARNING (ML): learns patterns from examples         │  │
│  │  ┌────────────────────────────────────────────────────────┐  │  │
│  │  │ DEEP LEARNING (DL): ML with large neural networks      │  │  │
│  │  │  ┌──────────────────────────────────────────────────┐  │  │  │
│  │  │  │ GENERATIVE AI: creates new text, images, code    │  │  │  │
│  │  │  │ (chat assistants, image generators)              │  │  │  │
│  │  │  └──────────────────────────────────────────────────┘  │  │  │
│  │  └────────────────────────────────────────────────────────┘  │  │
│  └──────────────────────────────────────────────────────────────┘  │
└────────────────────────────────────────────────────────────────────┘
```

- **Rules:** a person writes the decision (`if hours >= 10: ...`). Everything we did in Weeks 1–8.
- **Machine learning:** the computer *finds* the decision by looking at many examples with known answers.
- Every GenAI system is ML, but most ML is not GenAI. Next week: classic ML. Weeks 11–12: GenAI.

## 2. A hand-written rule

Our examples: 298 students. For each one we know their habits **and** whether they passed (`passed`: 1 = yes, 0 = no).

In [ ]:
import csv
from pathlib import Path

DATA_DIR = Path("../../../datasets")

with open(DATA_DIR / "students.csv", encoding="utf-8", newline="") as f:
    students = list(csv.DictReader(f))   # list() collects every row (a dict) into a list

print(len(students), "students")
print(students[0])

A teacher's rule of thumb: *"students who study at least 10 hours a week pass."*

We write it as a function with the **threshold** as a parameter, so we can try other numbers later.

In [ ]:
def predict_pass(student, threshold):
    """Predict 1 (pass) if the student studies at least `threshold` hours a week, else 0 (fail)."""
    if float(student["study_hours"]) >= threshold:   # CSV values are text: convert first!
        return 1
    return 0

🤔 **Predict:** the first student studies 14 hours. What does the rule predict? Did they actually pass?

In [ ]:
first = students[0]
print("Prediction:", predict_pass(first, 10), "| Actual:", first["passed"])

## 3. How good is the rule? Accuracy

**Accuracy** = correct predictions ÷ all predictions. We compare each prediction with the real answer.

In [ ]:
def accuracy(students, threshold):
    """Share of students (0 to 1) whose result the rule 'study_hours >= threshold' predicts correctly."""
    correct = 0
    for student in students:
        if predict_pass(student, threshold) == int(student["passed"]):   # "1" → 1
            correct += 1
    return correct / len(students)

In [ ]:
teacher_accuracy = accuracy(students, 10)
print(f"Teacher's rule (10 hours): {teacher_accuracy * 100:.1f}% correct")

🤔 **Discuss:** is 58.7 % good? What would you compare it with?

## 4. Learning the threshold from examples

Instead of *guessing* 10, let the data decide: try every threshold from 0 to 20 hours, measure each one, keep the best.

In [ ]:
best_threshold = 0
best_accuracy = 0

for threshold in range(0, 21):
    acc = accuracy(students, threshold)
    print(f"{threshold:>2} h  {acc * 100:5.1f}%  " + "#" * round(acc * 50))
    if acc > best_accuracy:          # better than anything so far? remember it
        best_accuracy = acc
        best_threshold = threshold

In [ ]:
print(f"Learned threshold: {best_threshold} hours → {best_accuracy * 100:.1f}% correct")

**That loop is training.** We tried many candidate rules on examples with known answers, measured each one and
kept the best. The learned number (4 hours) is our **model**.

Real ML libraries do the same thing (try, measure, improve), just much faster and with many columns at once.

## 5. Prediction: using the model on new students

Training happens once. **Prediction** happens every time a new case arrives. These students are *not* in the file.

In [ ]:
new_students = [
    {"name": "Omar", "study_hours": 6.5},
    {"name": "Mei", "study_hours": 2},
    {"name": "Diego", "study_hours": 4},
]
for student in new_students:
    print(student["name"], "→", predict_pass(student, best_threshold))

🤔 **Predict:** threshold 0 means "everyone passes". Look at the first line of the table in section 4. How accurate is
a "model" that simply says *pass* for every student?

In [ ]:
always_pass = accuracy(students, 0)
print(f"Always predict 'pass': {always_pass * 100:.1f}%")
print(f"Our learned rule:      {best_accuracy * 100:.1f}%")

71 % of students passed, so "always pass" is already right 71 % of the time. Our learned rule beats it by only
about 3 points. Lesson: **always compare with a simple baseline**. One column (study hours) is not enough. Next
week the computer combines many columns at once.

## 6. Is the model equally good for everyone?

🤔 **Predict:** is the learned rule as accurate for students **with** a part-time job as for those without?

In [ ]:
for group in ["no", "yes"]:
    members = []
    for student in students:
        if student["has_job"] == group:
            members.append(student)
    print(f"has_job={group:<3}  {len(members):>3} students  {accuracy(members, best_threshold) * 100:.1f}% correct")

The **same rule** is right for about 79 % of students without a job but only about 62 % of students with one.
Working students pass less often *at the same study time*, so the rule makes more mistakes for them.

The code is neutral; the pattern comes from the **data**. A model copies whatever patterns its examples contain,
including unfair ones. This is called **bias**.

💬 **Discuss:** a college wants to use this rule to (a) award scholarships, (b) offer free tutoring. Which use is
acceptable? What would you change?

### Rules, ML or GenAI?

| Use case | Best fit | Why |
|----------|----------|-----|
| Late fee = days late × 0.50 | rules | the rule is known and must be exact |
| Spam filter | ML | many examples exist; spam keeps changing |
| Predict next month's electricity use | ML | a pattern in past data |
| Summarise a long email | GenAI | creates new text |
| Check a password has at least 8 characters | rules | an exact policy |
| Unlock a phone with your face | DL | images; nobody can write "face rules" |

**When not to use AI:** the exact rule is known · there is little or no data · mistakes are very costly and no
human checks them · the decision must be fully explainable.

## Recap

1. AI ⊃ ML ⊃ DL ⊃ GenAI. ML **learns** patterns from examples instead of being given rules.
2. **Training** = try candidate rules on examples with known answers, measure, keep the best. **Prediction** = use it on new cases.
3. **Accuracy** = correct ÷ total. Always compare with a baseline, and check every group: a model can be biased.

**Next time:** the data itself. Features, labels, and the problems hidden in a messy raw file.